# DeepStream vs PyTorch — Energy & FPS Comparison

Compares YOLOv8n inference on a Jetson AGX Orin (MAXN) across:
- **PyTorch** pipeline: v4l2 → CPU decode → PyTorch CUDA inference
- **DeepStream** pipeline: v4l2 → GStreamer → nvinfer (TensorRT) inference

Both sweeps: imgsz ∈ {320, 640} × precision ∈ {fp16, fp32} × target_fps ∈ {0,5,10,15,20,25,30} × 3 repeats

Energy measured by INA3221 at 1 kHz across CPU/GPU/IO rails.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
RESULTS = ROOT / 'results' / 'camera_bench'

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Load data

In [ ]:
# ── PyTorch sweeps ────────────────────────────────────────────────────────────
pt640 = pd.read_csv(RESULTS / 'yolov8n_fps_sweep_MAXN_20260428_133405' / 'sweep_summary.csv')
pt640['yolo_imgsz'] = 640
pt640['backend'] = 'pytorch'

pt320 = pd.read_csv(RESULTS / 'yolov8n_fps_sweep_MAXN_20260428_150803' / 'sweep_summary.csv')
pt320['backend'] = 'pytorch'

pytorch = pd.concat([pt640, pt320], ignore_index=True)
pytorch = pytorch[pytorch['status'] == 'ok'].copy()

# ── DeepStream sweep ──────────────────────────────────────────────────────────
ds = pd.read_csv(RESULTS / 'deepstream_yolov8n_fps_sweep_MAXN_20260507_101832' / 'sweep_summary.csv')
ds = ds[ds['status'] == 'ok'].copy()

# Rename n_frames → n_timed for consistency
if 'n_frames' in ds.columns and 'n_timed' not in ds.columns:
    ds = ds.rename(columns={'n_frames': 'n_timed'})

# ── Combined ─────────────────────────────────────────────────────────────────
KEEP = ['backend','model','yolo_imgsz','precision','target_fps','repeat',
        'fps_mean','fps_p50','fps_p95','fps_min','fps_max',
        'energy_total_j','mean_power_w','energy_per_frame_j',
        'cpu_rail_j','gpu_rail_j','io_rail_j','duration_s']

combined = pd.concat(
    [pytorch[[c for c in KEEP if c in pytorch.columns]],
     ds[[c for c in KEEP if c in ds.columns]]],
    ignore_index=True
)
combined['precision'] = combined['precision'].str.upper()

print(f"PyTorch rows : {len(pytorch)}")
print(f"DeepStream rows: {len(ds)}")
print(f"Combined rows: {len(combined)}")
combined.groupby(['backend','yolo_imgsz','precision'])['target_fps'].count()

## 2. Aggregate (median over repeats)

In [ ]:
AGG_COLS = ['fps_mean','energy_per_frame_j','mean_power_w',
            'cpu_rail_j','gpu_rail_j','io_rail_j','energy_total_j']

agg = (
    combined
    .groupby(['backend','yolo_imgsz','precision','target_fps'])[AGG_COLS]
    .median()
    .reset_index()
)

agg['epf_mj'] = agg['energy_per_frame_j'] * 1000  # mJ/frame

# Savings table: deepstream vs pytorch
pt_agg = agg[agg.backend=='pytorch'].set_index(['yolo_imgsz','precision','target_fps'])
ds_agg = agg[agg.backend=='deepstream'].set_index(['yolo_imgsz','precision','target_fps'])

savings = pd.DataFrame(index=ds_agg.index)
savings['fps_pytorch']   = pt_agg['fps_mean']
savings['fps_deepstream']= ds_agg['fps_mean']
savings['epf_pt_mj']     = pt_agg['epf_mj']
savings['epf_ds_mj']     = ds_agg['epf_mj']
savings['epf_saving_pct']= (savings['epf_pt_mj'] - savings['epf_ds_mj']) / savings['epf_pt_mj'] * 100
savings['pwr_pt_w']      = pt_agg['mean_power_w']
savings['pwr_ds_w']      = ds_agg['mean_power_w']
savings['pwr_saving_pct']= (savings['pwr_pt_w'] - savings['pwr_ds_w']) / savings['pwr_pt_w'] * 100

savings = savings.reset_index()
print(savings[['yolo_imgsz','precision','target_fps',
               'epf_pt_mj','epf_ds_mj','epf_saving_pct',
               'pwr_pt_w','pwr_ds_w','pwr_saving_pct']].to_string(index=False, float_format='%.1f'))

## 3. Energy per frame vs target FPS

In [ ]:
IMSZ_LIST  = [640, 320]
PREC_LIST  = ['FP32', 'FP16']
BACKEND_STYLE = {
    'pytorch':    dict(color='#E87040', linestyle='-',  marker='o', label='PyTorch'),
    'deepstream': dict(color='#4C72B0', linestyle='--', marker='s', label='DeepStream (TRT)'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
fig.suptitle('Energy per Frame — DeepStream vs PyTorch (YOLOv8n, Jetson AGX Orin MAXN)', fontsize=13, y=1.01)

for row_i, imgsz in enumerate(IMSZ_LIST):
    for col_j, prec in enumerate(PREC_LIST):
        ax = axes[row_i][col_j]
        for backend, style in BACKEND_STYLE.items():
            sub = agg[(agg.backend==backend) &
                      (agg.yolo_imgsz==imgsz) &
                      (agg.precision==prec)].sort_values('target_fps')
            if sub.empty:
                continue
            ax.plot(sub['target_fps'], sub['epf_mj'],
                    **{k:v for k,v in style.items() if k!='label'},
                    label=style['label'], linewidth=2, markersize=6)

        ax.set_title(f'imgsz={imgsz}  {prec}', fontsize=11)
        ax.set_xlabel('Target FPS (0 = unbounded)')
        ax.set_ylabel('Energy / frame (mJ)')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'deepstream_vs_pytorch_epf.png', bbox_inches='tight')
plt.show()

## 4. Mean system power vs target FPS

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
fig.suptitle('Mean System Power — DeepStream vs PyTorch (YOLOv8n, Jetson AGX Orin MAXN)', fontsize=13, y=1.01)

for row_i, imgsz in enumerate(IMSZ_LIST):
    for col_j, prec in enumerate(PREC_LIST):
        ax = axes[row_i][col_j]
        for backend, style in BACKEND_STYLE.items():
            sub = agg[(agg.backend==backend) &
                      (agg.yolo_imgsz==imgsz) &
                      (agg.precision==prec)].sort_values('target_fps')
            if sub.empty:
                continue
            ax.plot(sub['target_fps'], sub['mean_power_w'],
                    **{k:v for k,v in style.items() if k!='label'},
                    label=style['label'], linewidth=2, markersize=6)

        ax.set_title(f'imgsz={imgsz}  {prec}', fontsize=11)
        ax.set_xlabel('Target FPS (0 = unbounded)')
        ax.set_ylabel('Mean system power (W)')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'deepstream_vs_pytorch_power.png', bbox_inches='tight')
plt.show()

## 5. Actual FPS achieved vs target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Actual FPS achieved vs Target FPS', fontsize=13)

for col_j, imgsz in enumerate(IMSZ_LIST):
    ax = axes[col_j]
    fps_range = [0, 32]
    ax.plot(fps_range, fps_range, 'k:', linewidth=1, label='ideal')

    for prec in PREC_LIST:
        for backend, style in BACKEND_STYLE.items():
            sub = agg[(agg.backend==backend) &
                      (agg.yolo_imgsz==imgsz) &
                      (agg.precision==prec)].sort_values('target_fps')
            if sub.empty:
                continue
            # Replace target_fps=0 with 30 for plotting (unbounded → camera max)
            x = sub['target_fps'].replace(0, 30)
            lbl = f"{backend} {prec}"
            ax.plot(x, sub['fps_mean'],
                    color=style['color'],
                    linestyle='-' if prec=='FP32' else ':',
                    marker=style['marker'],
                    label=lbl, linewidth=1.5, markersize=5)

    ax.set_title(f'imgsz={imgsz}')
    ax.set_xlabel('Target FPS (0→plotted as 30)')
    ax.set_ylabel('Actual FPS')
    ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

## 6. Rail breakdown: GPU vs CPU vs IO

In [ ]:
# Stacked bar: GPU / CPU / IO energy per frame at unbounded FPS (target_fps=0)
sub = agg[agg.target_fps == 0].copy()
sub['cpu_mj']  = sub['cpu_rail_j'] / sub['energy_total_j'] * sub['epf_mj']
sub['gpu_mj']  = sub['gpu_rail_j'] / sub['energy_total_j'] * sub['epf_mj']
sub['io_mj']   = sub['io_rail_j']  / sub['energy_total_j'] * sub['epf_mj']
sub['label']   = sub.apply(lambda r: f"{r['backend']}\nimgsz{int(r['yolo_imgsz'])} {r['precision']}", axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(sub))
w = 0.6
ax.bar(x, sub['gpu_mj'], w, label='GPU rail', color='#4C72B0')
ax.bar(x, sub['cpu_mj'], w, bottom=sub['gpu_mj'], label='CPU rail', color='#DD8452')
ax.bar(x, sub['io_mj'],  w, bottom=sub['gpu_mj']+sub['cpu_mj'], label='IO rail', color='#55A868')

ax.set_xticks(x)
ax.set_xticklabels(sub['label'], fontsize=9)
ax.set_ylabel('Energy per frame (mJ)')
ax.set_title('Energy per Frame by Rail — Unbounded FPS (target_fps=0)')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'deepstream_vs_pytorch_rail_breakdown.png', bbox_inches='tight')
plt.show()

## 7. Summary table

In [ ]:
# Pivot: mean energy/frame (mJ) and mean power (W) at unbounded FPS
tbl = savings[savings.target_fps == 0][[
    'yolo_imgsz','precision',
    'fps_pytorch','fps_deepstream',
    'epf_pt_mj','epf_ds_mj','epf_saving_pct',
    'pwr_pt_w','pwr_ds_w','pwr_saving_pct'
]].rename(columns={
    'yolo_imgsz': 'imgsz',
    'fps_pytorch': 'fps_PT',
    'fps_deepstream': 'fps_DS',
    'epf_pt_mj': 'mJ/frame PT',
    'epf_ds_mj': 'mJ/frame DS',
    'epf_saving_pct': 'saving %',
    'pwr_pt_w': 'W PT',
    'pwr_ds_w': 'W DS',
    'pwr_saving_pct': 'pwr saving %',
})

pd.set_option('display.float_format', '{:.1f}'.format)
display(tbl.reset_index(drop=True))

## 8. Energy per frame at all FPS targets — heatmap

In [ ]:
import matplotlib.colors as mcolors

configs = [(640,'FP32'),(640,'FP16'),(320,'FP32'),(320,'FP16')]
fps_targets = sorted(agg['target_fps'].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_i, backend in enumerate(['pytorch','deepstream']):
    ax = axes[ax_i]
    mat = []
    for imgsz, prec in configs:
        row_data = []
        for fps in fps_targets:
            v = agg[(agg.backend==backend)&(agg.yolo_imgsz==imgsz)&
                    (agg.precision==prec)&(agg.target_fps==fps)]['epf_mj']
            row_data.append(v.values[0] if len(v) else np.nan)
        mat.append(row_data)
    mat = np.array(mat)

    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn_r')
    ax.set_xticks(range(len(fps_targets)))
    ax.set_xticklabels(fps_targets)
    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels([f'imgsz{i} {p}' for i,p in configs])
    ax.set_xlabel('Target FPS')
    ax.set_title(f'{backend.title()} — mJ/frame')
    plt.colorbar(im, ax=ax, label='mJ/frame')

    for r in range(len(configs)):
        for c in range(len(fps_targets)):
            v = mat[r, c]
            if not np.isnan(v):
                ax.text(c, r, f'{v:.0f}', ha='center', va='center', fontsize=8,
                        color='white' if v > mat.max()*0.7 else 'black')

plt.suptitle('Energy per Frame Heatmap (mJ)', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'deepstream_vs_pytorch_heatmap.png', bbox_inches='tight')
plt.show()